# Step 1: Initializing the Database and Inserting the data

## Importing the Required Libraries

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
import json

### Initializing the Embedding Model

#### Right now, I am using a fast, lightweight model for prototyping. During evaluation Phase of the project, I am planning to swap it with 'BAAI/bge-m3' to compare the performance, and possibily benchmark too.

In [2]:
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 2. Initializing ChromaDB (Saving it to vector_store folder (locally))

In [3]:
print("Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(path="../data/vector_store")
collection = chroma_client.get_or_create_collection(name="legal_clauses")

Initializing ChromaDB...


#### Creating a function that takes the JSON payload from the extractor, embeds the text, and saves it into ChromaDB

In [4]:
def index_clause(payload):
    """
    Takes the JSON payload from the unified extractor, embeds the text, 
    and saves it into ChromaDB.
    """
    # Generate the vector embedding for the enriched text
    vector = embedder.encode(payload["enriched_text"]).tolist()
    
    # Insert into ChromaDB
    collection.add(
        ids=[str(payload["id"])],
        embeddings=[vector],
        documents=[payload["enriched_text"]],
        metadatas=[payload["metadata"]]
    )
    print(f"✅ Successfully indexed clause ID: {payload['id']}")


## TESTING IT OUT

In [5]:
# Creating two dummy processed payloads to index
payload_1 = {
    "id": "1001",
    "enriched_text": "[Article 5: Usage Fees] The User shall pay the usage fee calculated by the Company, subject to Section 8.1.",
    "metadata": {
        "parent_scope": "Article 5: Usage Fees",
        "obligations": "shall pay the usage fee",
        "permissions": "",
        "conditions": "subject to Section 8.1",
        "linked_nodes": "Linked Node: 8.1"
    }
}

payload_2 = {
    "id": "1002",
    "enriched_text": "[Article 8: Refunds] Section 8.1: Usage fees are strictly non-refundable once the service has been accessed.",
    "metadata": {
        "parent_scope": "Article 8: Refunds",
        "obligations": "",
        "permissions": "",
        "conditions": "once the service has been accessed",
        "linked_nodes": ""
    }
}

# Index theming into the database
index_clause(payload_1)
index_clause(payload_2)

✅ Successfully indexed clause ID: 1001
✅ Successfully indexed clause ID: 1002


# Step 2: Testing the Semantic Search

In [6]:
# Simulating a user query
user_query = "Are there any conditions where I have to pay fees?"

# Embedding the user query
query_vector = embedder.encode(user_query).tolist()

# Searching the database for the top 1 most relevant clause
results = collection.query(
    query_embeddings=[query_vector],
    n_results=1
)

In [7]:
print("\n🔍 Search Results:")
print("Matched Document:", results["documents"][0][0])
print("Attached Metadata:", json.dumps(results["metadatas"][0][0], indent=4))


🔍 Search Results:
Matched Document: [Article 8: Refunds] Section 8.1: Usage fees are strictly non-refundable once the service has been accessed.
Attached Metadata: {
    "permissions": "",
    "obligations": "",
    "linked_nodes": "",
    "parent_scope": "Article 8: Refunds",
    "conditions": "once the service has been accessed"
}


##### The Expected Output was Article 5 (Usage Fees) which dictates payment rules, but instead it gave Article 8 (Refunds) which discusses non-refundable usage fees, as that was mathematically closer to the question upon seeing the word "fees" for the lightweight embedding model (all-MiniLM-L6-v2)

##### This phenomenon is known as **"dense retriver blindness"**. It seems to occur because of general-purpose, lightweight dense embedding models (like `all-MiniLM-L6-v2`) **prioritize raw semantic overlap over structural or logical precision**. 

##### This proves why naïve, single-pass semantic RAG is insufficient for legal documents. 

##### This validates the necessity of the **makeLEGalWORKeasy** architecture: by structuring text into Scope-Stacks and explicitly tagging dependencies (e.g., "Linked Node: 8.1") in the metadata, the system is no longer solely reliant on flawed vector similarity to piece together legal logic.

## Using a query specifically targeting the payment rule to trigger the linked node

In [8]:
def single_pass_retriever(user_query):
    """
    Retrieves the primary clause and its linked reference in one go.
    """
    print(f"User Query: '{user_query}'\n")
    
    # 1. Fetch Primary Clause
    query_vector = embedder.encode(user_query).tolist()
    primary_result = collection.query(
        query_embeddings=[query_vector],
        n_results=1
    )
    
    primary_doc = primary_result["documents"][0][0]
    metadata = primary_result["metadatas"][0][0]
    
    # Initialize the context window that will be sent to the LLM
    context_window = [f"PRIMARY RULE:\n{primary_doc}"]
    
    # 2. Check for 1-Hop Linked Dependencies
    linked_nodes_str = metadata.get("linked_nodes", "")
    
    if linked_nodes_str:
        print(f"🔗 Dependency Detected: {linked_nodes_str}")
        print("Fetching linked exception...\n")
        
        # Clean the string to search the DB for the referenced section
        link_target = linked_nodes_str.replace("Linked Node: ", "Section ")
        link_vector = embedder.encode(link_target).tolist()
        
        # Fetch the linked node
        link_result = collection.query(
            query_embeddings=[link_vector],
            n_results=1
        )
        
        linked_doc = link_result["documents"][0][0]
        context_window.append(f"LINKED EXCEPTION:\n{linked_doc}")
    else:
        print("✅ No dependencies found. Rule is standalone.\n")
        
    # 3. Combine into the final de-polluted prompt
    final_prompt_context = "\n\n".join(context_window)
    
    return final_prompt_context

## TESTING IT OUT

In [9]:
test_query = "What happens with the usage fee calculation?"
final_context = single_pass_retriever(test_query)

print("========================================")
print("FINAL DE-POLLUTED LLM CONTEXT WINDOW:")
print("========================================")
print(final_context)

User Query: 'What happens with the usage fee calculation?'

🔗 Dependency Detected: Linked Node: 8.1
Fetching linked exception...

FINAL DE-POLLUTED LLM CONTEXT WINDOW:
PRIMARY RULE:
[Article 5: Usage Fees] The User shall pay the usage fee calculated by the Company, subject to Section 8.1.

LINKED EXCEPTION:
[Article 5: Usage Fees] The User shall pay the usage fee calculated by the Company, subject to Section 8.1.


#### From the looks of it, the RAG seems to have stumbled upon another phenomenal failure- the failure of Semantic Dependency Resolution. 

During the 1-hop reference jump, the system initially extracted the link "Section 8.1" and attempted to fetch it using semantic vector search ("collection.query"). 

But the system returned the primary clause (Article 5) a second time instead of fetching the target clause (Article 8.1). 

Because the primary clause contains the explicit text "subject to Section 8.1", its embedding vector was mathematically closer to the search query "Section 8.1" than the actual target clause. This proves that **semantic similarity cannot reliably resolve explicit legal pointers**.

To prevent recursive loops, the 1-hop linker must bypass the embedding model entirely. I plan to replace semantic search ("collection.query") with deterministic string filtering ("collection.get(where_document=...)") to ensure the system strictly fetches the defining target node.

## Detriministic 1-Hop Jump

In [10]:
def single_pass_retriever(user_query):
    """
    Retrieves the primary clause and its linked reference in one go.
    """
    print(f"User Query: '{user_query}'\n")
    
    # 1. Fetch Primary Clause (Semantic Search)
    query_vector = embedder.encode(user_query).tolist()
    primary_result = collection.query(
        query_embeddings=[query_vector],
        n_results=1
    )
    
    primary_doc = primary_result["documents"][0][0]
    metadata = primary_result["metadatas"][0][0]
    
    # Initialize the context window that will be sent to the LLM
    context_window = [f"PRIMARY RULE:\n{primary_doc}"]
    
    # 2. Check for 1-Hop Linked Dependencies
    linked_nodes_str = metadata.get("linked_nodes", "")
    
    if linked_nodes_str:
        print(f"🔗 Dependency Detected: {linked_nodes_str}")
        print("Fetching linked exception DETERMINISTICALLY...\n")
        
        # Clean the string to get just the section number (e.g., "8.1")
        target_section = linked_nodes_str.replace("Linked Node: ", "").strip()
        
        # DETERMINISTIC RETRIEVAL: Bypass embeddings completely.
        # Force the database to find the text where the section is actually defined.
        # We look for "Section 8.1:" (with the colon) to avoid catching mere mentions.
        link_result = collection.get(
            where_document={"$contains": f"Section {target_section}:"}
        )
        
        # .get() returns a flat list for documents, not a nested list like .query()
        if link_result["documents"]:
            linked_doc = link_result["documents"][0]
            context_window.append(f"LINKED EXCEPTION:\n{linked_doc}")
        else:
            context_window.append(f"LINKED EXCEPTION:\n[Target Section {target_section} not found]")
    else:
        print("✅ No dependencies found. Rule is standalone.\n")
        
    # 3. Combine into the final de-polluted prompt
    final_prompt_context = "\n\n".join(context_window)
    
    return final_prompt_context

## TESTING IT OUT

In [11]:
test_query = "What happens with the usage fee calculation?"
final_context = single_pass_retriever(test_query)

print("========================================")
print("FINAL DE-POLLUTED LLM CONTEXT WINDOW:")
print("========================================")
print(final_context)

User Query: 'What happens with the usage fee calculation?'

🔗 Dependency Detected: Linked Node: 8.1
Fetching linked exception DETERMINISTICALLY...

FINAL DE-POLLUTED LLM CONTEXT WINDOW:
PRIMARY RULE:
[Article 5: Usage Fees] The User shall pay the usage fee calculated by the Company, subject to Section 8.1.

LINKED EXCEPTION:
[Article 8: Refunds] Section 8.1: Usage fees are strictly non-refundable once the service has been accessed.
